# Small-lesion threshold diagnostics
H-N00과 H-N50의 best-composite checkpoint를 사용해 PHYS_58, PHYS_82, PHYS_78의 전처리 후 병변 크기와 probability threshold별 Dice/Recall/FP를 진단합니다. 이 분석은 preprocessed grid에서 수행되므로 공식 full-volume Dice와 수치가 조금 다를 수 있습니다.

In [1]:
# 1. imports와 경로
import gc
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from scipy.ndimage import label

from brainon_ai.nnunet.runner import configure_paths
from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
from nnunetv2.training.dataloading.nnunet_dataset import nnUNetDatasetBlosc2

ROOT = Path('/home/stu03/workspace/model').resolve()
DATA_ROOT = Path('/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet').resolve()
PREPROCESSED_ROOT = DATA_ROOT / 'nnUNet_preprocessed'
RESULTS_ROOT = ROOT / 'runs/nnunet'
OUTPUT_DIR = ROOT / 'runs/analysis/small_lesion_threshold_diagnostics'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CASES = ['PHYS_58', 'PHYS_82', 'PHYS_78']
THRESHOLDS = [0.05, 0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
print('Output:', OUTPUT_DIR)

Output: /home/stu03/workspace/model/runs/analysis/small_lesion_threshold_diagnostics


In [2]:
# 2. 모델과 전처리 데이터 설정
MODELS = {
    'H-N00': {
        'config': ROOT / 'configs/experiments/hemorrhage/H-N00.yaml',
        'model_folder': RESULTS_ROOT / 'Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansBaselineV1__3d_fullres',
        'data_folder': PREPROCESSED_ROOT / 'Dataset001_ICHBinary/nnUNetPlans_3d_fullres',
    },
    'H-N50': {
        'config': ROOT / 'configs/experiments/hemorrhage/H-N50.yaml',
        'model_folder': RESULTS_ROOT / 'Dataset003_ICHBinary2Window/nnUNetTrainerICHBaseline__ICHPlansTwoWindowBaselineV1__3d_fullres',
        'data_folder': PREPROCESSED_ROOT / 'Dataset003_ICHBinary2Window/nnUNetPlans_3d_fullres',
    },
}
for experiment, paths in MODELS.items():
    assert paths['config'].is_file(), paths['config']
    assert paths['data_folder'].is_dir(), paths['data_folder']
    checkpoint = paths['model_folder'] / 'fold_0/checkpoint_best_composite.pth'
    assert checkpoint.is_file(), checkpoint
    print(experiment, 'checkpoint:', checkpoint)

H-N00 checkpoint: /home/stu03/workspace/model/runs/nnunet/Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansBaselineV1__3d_fullres/fold_0/checkpoint_best_composite.pth
H-N50 checkpoint: /home/stu03/workspace/model/runs/nnunet/Dataset003_ICHBinary2Window/nnUNetTrainerICHBaseline__ICHPlansTwoWindowBaselineV1__3d_fullres/fold_0/checkpoint_best_composite.pth


In [3]:
# 3. 전처리 후 병변 voxel, z-slice, component 확인
lesion_rows = []
for experiment, paths in MODELS.items():
    dataset = nnUNetDatasetBlosc2(str(paths['data_folder']), CASES)
    for case_id in CASES:
        data, segmentation, _, properties = dataset.load_case(case_id)
        mask = np.asarray(segmentation)[0] > 0
        components, count = label(mask)
        sizes = [int((components == index).sum()) for index in range(1, count + 1)]
        positive_slices = np.where(np.any(mask, axis=(1, 2)))[0]
        lesion_rows.append({
            'experiment': experiment, 'case_id': case_id,
            'input_channels': int(data.shape[0]), 'shape': tuple(mask.shape),
            'positive_voxels': int(mask.sum()),
            'positive_z_slices': int(len(positive_slices)),
            'z_indices': positive_slices.tolist(),
            'components': int(count),
            'largest_component': max(sizes) if sizes else 0,
            'source_spacing': properties.get('spacing'),
        })
lesion_table = pd.DataFrame(lesion_rows)
lesion_table.to_csv(OUTPUT_DIR / 'preprocessed_lesion_statistics.csv', index=False)
display(lesion_table)

,experiment,case_id,input_channels,shape,positive_voxels,positive_z_slices,z_indices,components,largest_component,source_spacing
0,H-N00,PHYS_58,1,"(38, 446, 446)",204,2,"[12, 13]",1,204,"[5.0, 0.42578125, 0.42578125]"
1,H-N00,PHYS_78,1,"(36, 432, 432)",4791,11,"[10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21]",2,3995,"[5.0, 0.412109375, 0.412109375]"
2,H-N00,PHYS_82,1,"(34, 430, 430)",1437,2,"[9, 10]",1,1437,"[5.0, 0.41015625, 0.41015625]"
3,H-N50,PHYS_58,2,"(38, 446, 446)",204,2,"[12, 13]",1,204,"[5.0, 0.42578125, 0.42578125]"
4,H-N50,PHYS_78,2,"(34, 432, 432)",4791,11,"[10, 11, 12, 13, 14, 15, 16, 18, 19, 20, 21]",2,3995,"[5.0, 0.412109375, 0.412109375]"
5,H-N50,PHYS_82,2,"(32, 415, 399)",1422,2,"[9, 10]",1,1422,"[5.0, 0.41015625, 0.41015625]"


In [4]:
# 4. threshold 지표 함수
def threshold_metrics(probability, reference, threshold):
    prediction = probability >= threshold
    reference = reference > 0
    tp = int(np.logical_and(prediction, reference).sum())
    fp = int(np.logical_and(prediction, ~reference).sum())
    fn = int(np.logical_and(~prediction, reference).sum())
    denominator = 2 * tp + fp + fn
    return {
        'threshold': float(threshold),
        'dice': 2 * tp / denominator if denominator else float('nan'),
        'recall': tp / (tp + fn) if tp + fn else float('nan'),
        'precision': tp / (tp + fp) if tp + fp else 0.0,
        'tp': tp, 'fp': fp, 'fn': fn,
        'predicted_voxels': int(prediction.sum()),
        'detected': bool(tp > 0),
    }

In [5]:
# 5. checkpoint 추론과 threshold sweep — 약간 시간이 걸립니다
configure_paths(DATA_ROOT, RESULTS_ROOT)
threshold_rows = []
probability_maps = {}
reference_maps = {}
input_maps = {}

for experiment, paths in MODELS.items():
    print('Loading:', experiment)
    os.environ['ICH_NNUNET_CONFIG'] = str(paths['config'])
    predictor = nnUNetPredictor(
        tile_step_size=0.5, use_gaussian=True, use_mirroring=True,
        perform_everything_on_device=True, device=torch.device('cuda'),
        verbose=False, verbose_preprocessing=False, allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(paths['model_folder']), use_folds=(0,),
        checkpoint_name='checkpoint_best_composite.pth',
    )
    dataset = nnUNetDatasetBlosc2(str(paths['data_folder']), CASES)
    for case_id in CASES:
        print('Predicting:', experiment, case_id)
        data, segmentation, _, _ = dataset.load_case(case_id)
        data_array = np.asarray(data, dtype=np.float32)
        reference = np.asarray(segmentation)[0] > 0
        tensor = torch.from_numpy(data_array).to(dtype=torch.float32, memory_format=torch.contiguous_format)
        with torch.no_grad():
            logits = predictor.predict_logits_from_preprocessed_data(tensor)
            probabilities = torch.softmax(logits, dim=0)
        probability = probabilities[1].cpu().numpy()
        gt_probabilities = probability[reference]
        probability_maps[(experiment, case_id)] = probability
        reference_maps[(experiment, case_id)] = reference
        input_maps[(experiment, case_id)] = data_array[0]
        for threshold in THRESHOLDS:
            threshold_rows.append({
                'experiment': experiment, 'case_id': case_id,
                'gt_probability_max': float(gt_probabilities.max()),
                'gt_probability_mean': float(gt_probabilities.mean()),
                'gt_probability_p95': float(np.percentile(gt_probabilities, 95)),
                **threshold_metrics(probability, reference, threshold),
            })
        del tensor, logits, probabilities
        torch.cuda.empty_cache()
    del predictor
    gc.collect()
    torch.cuda.empty_cache()

threshold_results = pd.DataFrame(threshold_rows)
threshold_results.to_csv(OUTPUT_DIR / 'threshold_metrics.csv', index=False)
display(threshold_results)

Loading: H-N00


/home/stu03/workspace/model/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Trainer 'nnUNetTrainerICHBaseline' not found in nnunetv2.training.nnUNetTrainer.
Searching in external trainer paths from environment variable 'nnUNet_extTrainer'...
Searching in: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Searching for class nnUNetTrainerICHBaseline in folder /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers with current module None
  Inspecting module: configurable_trainer
Found class nnUNetTrainerICHBaseline in configurable_trainer
Using trainer 'nnUNetTrainerICHBaseline' from: /home/stu03/workspace/model/src/brainon_ai/nnunet/trainers
Predicting: H-N00 PHYS_58


100%|██████████| 18/18 [00:02<00:00,  6.99it/s]


Predicting: H-N00 PHYS_78


...

Predicting: H-N50 PHYS_78


100%|██████████| 18/18 [00:01<00:00,  9.26it/s]


Predicting: H-N50 PHYS_82


100%|██████████| 18/18 [00:01<00:00,  9.28it/s]


,experiment,case_id,gt_probability_max,gt_probability_mean,gt_probability_p95,threshold,dice,recall,precision,tp,fp,fn,predicted_voxels,detected
0,H-N00,PHYS_58,0.000013,0.000007,0.000011,0.05,0.000000,0.000000,0.000000,0,1164,204,1164,False
1,H-N00,PHYS_58,0.000013,0.000007,0.000011,0.10,0.000000,0.000000,0.000000,0,999,204,999,False
2,H-N00,PHYS_58,0.000013,0.000007,0.000011,0.20,0.000000,0.000000,0.000000,0,843,204,843,False
3,H-N00,PHYS_58,0.000013,0.000007,0.000011,0.30,0.000000,0.000000,0.000000,0,742,204,742,False
4,H-N00,PHYS_58,0.000013,0.000007,0.000011,0.40,0.000000,0.000000,0.000000,0,663,204,663,False
5,H-N00,PHYS_58,0.000013,0.000007,0.000011,0.50,0.000000,0.000000,0.000000,0,601,204,601,False
6,H-N00,PHYS_58,0.000013,0.000007,0.000011,0.60,0.000000,0.000000,0.000000,0,554,204,554,False
7,H-N00,PHYS_58,0.000013,0.000007,0.000011,0.70,0.000000,0.000000,0.000000,0,513,204,513,False
8,H-N00,PHYS_78,0.998047,0.140991,0.995605,0.05,0.222549,0.221248,0.223865,1060,3675,3731,4735,True
9,H-N00,PHYS_78,0.998047,0.140991,0.995605,0.10,0.204117,0.195575,0.213440,937,3453,3854,4390,True


In [6]:
# 6. GT 위치 probability 요약
probability_summary = (
    threshold_results.groupby(['experiment', 'case_id'])
    .agg(
        gt_probability_max=('gt_probability_max', 'first'),
        gt_probability_mean=('gt_probability_mean', 'first'),
        gt_probability_p95=('gt_probability_p95', 'first'),
    )
    .reset_index()
)
probability_summary.to_csv(OUTPUT_DIR / 'gt_probability_summary.csv', index=False)
display(probability_summary)

,experiment,case_id,gt_probability_max,gt_probability_mean,gt_probability_p95
0,H-N00,PHYS_58,0.000013,0.000007,0.000011
1,H-N00,PHYS_78,0.998047,0.140991,0.995605
2,H-N00,PHYS_82,0.000010,0.000003,0.000006
3,H-N50,PHYS_58,0.000106,0.000037,0.000091
4,H-N50,PHYS_78,1.000000,0.215454,0.998535
5,H-N50,PHYS_82,0.014473,0.000229,0.000864


In [7]:
# 7. threshold별 Dice와 Recall 곡선
for case_id in CASES:
    subset = threshold_results[threshold_results.case_id == case_id]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for experiment in MODELS:
        values = subset[subset.experiment == experiment].sort_values('threshold')
        axes[0].plot(values.threshold, values.dice, marker='o', label=experiment)
        axes[1].plot(values.threshold, values.recall, marker='o', label=experiment)
        axes[2].plot(values.threshold, values.fp, marker='o', label=experiment)
    for axis, title, ylabel in zip(axes, ['Dice', 'Recall', 'False-positive voxels'], ['Dice', 'Recall', 'FP voxels']):
        axis.set_title(f'{case_id}: {title}')
        axis.set_xlabel('Probability threshold')
        axis.set_ylabel(ylabel)
        axis.grid(alpha=0.25)
        axis.legend()
    axes[0].set_ylim(-0.03, 1.03)
    axes[1].set_ylim(-0.03, 1.03)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'{case_id}_threshold_curves.png', dpi=160)
    plt.show()

In [8]:
# 8. GT 최대 slice에서 probability map 확인
for case_id in CASES:
    fig, axes = plt.subplots(len(MODELS), 3, figsize=(13, 4 * len(MODELS)))
    for row, experiment in enumerate(MODELS):
        reference = reference_maps[(experiment, case_id)]
        probability = probability_maps[(experiment, case_id)]
        image = input_maps[(experiment, case_id)]
        z = int(reference.reshape(reference.shape[0], -1).sum(axis=1).argmax())
        axes[row, 0].imshow(image[z], cmap='gray')
        axes[row, 0].contour(reference[z], levels=[0.5], colors='lime', linewidths=1)
        axes[row, 0].set_title(f'{experiment} input + GT')
        heat = axes[row, 1].imshow(probability[z], cmap='magma', vmin=0, vmax=1)
        axes[row, 1].contour(reference[z], levels=[0.5], colors='lime', linewidths=1)
        axes[row, 1].set_title('Hemorrhage probability')
        axes[row, 2].imshow(probability[z] >= 0.5, cmap='gray', vmin=0, vmax=1)
        axes[row, 2].contour(reference[z], levels=[0.5], colors='lime', linewidths=1)
        axes[row, 2].set_title('Prediction at threshold 0.5')
        for axis in axes[row]:
            axis.axis('off')
        fig.colorbar(heat, ax=axes[row, 1], fraction=0.046, pad=0.04)
    fig.suptitle(f'{case_id} — green contour is GT', y=1.01)
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / f'{case_id}_probability_maps.png', dpi=160, bbox_inches='tight')
    plt.show()

In [9]:
# 9. 각 모델·case에서 Dice가 최대인 진단 threshold
best_thresholds = (
    threshold_results.sort_values(['experiment', 'case_id', 'dice'], ascending=[True, True, False])
    .groupby(['experiment', 'case_id'], as_index=False)
    .first()
)
best_thresholds.to_csv(OUTPUT_DIR / 'best_diagnostic_thresholds.csv', index=False)
display(best_thresholds[['experiment', 'case_id', 'threshold', 'dice', 'recall', 'precision', 'fp', 'fn', 'detected']])
print('주의: 이 표의 threshold는 세 case의 실패 원인 진단용이며 최종 threshold 선택값이 아닙니다.')

,experiment,case_id,threshold,dice,recall,precision,fp,fn,detected
0,H-N00,PHYS_58,0.05,0.000000,0.000000,0.000000,1164,204,False
1,H-N00,PHYS_78,0.05,0.222549,0.221248,0.223865,3675,3731,True
2,H-N00,PHYS_82,0.05,0.000000,0.000000,0.000000,320,1437,False
3,H-N50,PHYS_58,0.05,0.000000,0.000000,0.000000,1457,204,False
4,H-N50,PHYS_78,0.05,0.302691,0.299311,0.306149,3250,3357,True
5,H-N50,PHYS_82,0.05,0.000000,0.000000,0.000000,976,1422,False


주의: 이 표의 threshold는 세 case의 실패 원인 진단용이며 최종 threshold 선택값이 아닙니다.
